# Customer Segmentation using RFM + K-Means
This notebook analyzes 5,000 synthetic customers, creates RFM features, evaluates K-Means clustering, and generates business-oriented segment insights.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score


In [ ]:
df = pd.read_csv('../data/customers.csv')
df.head()


In [ ]:
rfm = df[['CustomerID','RecencyDays','PurchaseFrequency','MonetaryValue']].copy()
rfm.columns = ['CustomerID','Recency','Frequency','Monetary']
rfm.describe()


In [ ]:
X = np.log1p(rfm[['Recency','Frequency','Monetary']])
X = StandardScaler().fit_transform(X)
scores = {}
for k in range(2, 9):
    model = KMeans(n_clusters=k, random_state=42, n_init=20)
    labels = model.fit_predict(X)
    scores[k] = silhouette_score(X, labels)
scores


In [ ]:
best_k = max(scores, key=scores.get)
model = KMeans(n_clusters=best_k, random_state=42, n_init=20)
rfm['Cluster'] = model.fit_predict(X)
rfm.groupby('Cluster').agg(Customers=('CustomerID','count'), AvgRecency=('Recency','mean'), AvgFrequency=('Frequency','mean'), AvgMonetary=('Monetary','mean')).round(2)


## Business interpretation
- Low recency + high frequency + high monetary value → high-value/Champions-like customers.
- High recency + low frequency → reactivation opportunity.
- Mid-range customers can be targeted with loyalty and cross-sell campaigns.
- Segment names should be assigned after inspecting cluster profiles rather than assuming cluster IDs have meaning.

In [ ]:
summary = rfm.groupby('Cluster').agg(Customers=('CustomerID','count'), AvgRecency=('Recency','mean'), AvgFrequency=('Frequency','mean'), AvgMonetary=('Monetary','mean')).round(2)
summary.sort_values('AvgMonetary', ascending=False)
